### **Student Information**
GitHub ID: Mandy745-max

Kaggle name: olpolpolp

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**

---

## 1. Model Development (10 pts Required)

### 1.1 Preprocessing Steps

**The data.** Kaggle gave 4 files:
- `final_posts.json`: 4,800 posts (`post_id`, `text`)
- `data_identification.csv`: says if each post is `train` or `test`
- `emotion.csv`: the emotion label of each train post
- `sampleSubmission.csv`: the format and order of the answer file

**What I did:**
1. **Put the files together (code section 1).** I turned the json into a table and joined it with the other files by `post_id`.
   Result: **3,360 train posts** (with labels) and **1,440 test posts** (no labels).
2. **Checked the data (code section 1).**
   - Empty posts: **0**.
   - The 6 labels are not balanced: joy 875, anger 674, sadness 628, disgust 454, surprise 441, fear 288.
     Fear has only about 1/3 as many posts as joy. I score my experiments with **macro F1** (`average="macro"` or `scoring="f1_macro"` in my code), so a small class like fear counts as much as joy.
   - Post length (all 4,800 posts): 16.3 words on average, 75% have 19 words or fewer, and the longest has 510 words.
3. **I did not clean the text myself.**
   - TF-IDF uses its default settings: it changes words to lowercase and keeps only words with 2 or more letters or numbers, so it ignores punctuation and emojis.
   - The transformer models get the raw text. Each model's own tokenizer cuts it into small pieces (tokens), and I keep at most 128 tokens per post (`max_length=128`). Longer posts lose their end.
4. **Answer file (code section 3).** I keep the same posts and the same order as `sampleSubmission.csv`.
   Kaggle said "ID column id not found", so I renamed the column `post_id` to `id`.

### 1.2 Feature Engineering Steps

I used two kinds of features.

**A. Features I made myself (section 3.1, on my laptop).**
I fed them to a Linear SVM and scored them with 5-fold cross-validation (mean macro F1).
TF-IDF learns its words only from the training part of each fold, so it never sees the fold it is tested on.

| Features | 5-fold CV |
|---|---|
| TF-IDF, `min_df=2` (the Master's setting) | 0.4625 |
| TF-IDF, `min_df=1` (keep rare words too) | 0.4768 |
| TF-IDF, `min_df=1` + 2-word phrases | 0.4860 |
| MiniLM sentence embeddings (384 numbers per post) | 0.5003 |
| **TF-IDF (1-2 words) + MiniLM, side by side** | **0.5313** -> my first upload, sub1, Public 0.5321 |
| mpnet sentence embeddings (768 numbers per post) | 0.4806 |
| TF-IDF (1-2 words) + mpnet | 0.5278 |
| TF-IDF (1-2 words) + MiniLM + mpnet | 0.5359 |

- TF-IDF and MiniLM together scored higher than each one alone (0.5313 vs 0.4860 and 0.5003).
- Adding mpnet raised the score by only 0.0046. This is smaller than how much the score moves from fold to fold (std 0.0206 without mpnet, 0.0118 with mpnet). So I did not use it.

**B. Features learned by the model (fine-tuning, sections 3.1-3.3).**
With fine-tuning, I do not make features myself. The tokenizer turns the raw text into tokens, and the transformer learns its own features while it learns the 6 emotions.
On the same 80/20 split, fine-tuned distilroberta scored **0.6166**, and my sub1 features (TF-IDF + MiniLM + SVM) scored **0.5375**.
So from here on, all my models were fine-tuned transformers.

### 1.3 Explanation of Your Model

**My final model is an ensemble: I add up the probabilities of several fine-tuned transformer models and pick the emotion with the highest total.**

**How one model is trained.** I start from a pre-trained model and add a new layer that outputs the 6 emotions. Then I train the whole model on our posts.
- Same for all models: AdamW optimizer (weight decay 0.01), batches of 16 posts, 3 epochs, up to 128 tokens.
- Base-size models: learning rate 3e-5.
- Large models: a gentler setting (I call it the "safe recipe" in my code):
  learning rate 1e-5, warmup (the learning rate grows slowly in the first 10% of steps), and gradient clipping at 1.0 (no huge updates).

**5-fold training.** Each model is trained 5 times, once per fold. For the test posts I average the 5 models.
For the train posts I get an out-of-fold (OOF) prediction, so I can score my ideas without Kaggle. Fine-tuning was too slow on my laptop (about 6 to 8 minutes per epoch), so I ran this on Kaggle's GPU.

**The models (OOF macro F1, each with 5 folds):**

| Short name | Model | What it is | OOF |
|---|---|---|---|
| d | `distilroberta-base` | small RoBERTa | 0.5784 |
| r | `roberta-base` | RoBERTa, base size | 0.5879 |
| t | `cardiffnlp/twitter-roberta-base` | RoBERTa that also learned from tweets | 0.5907 |
| L / L2 | `roberta-large` | big RoBERTa (L2 = trained again with other seeds) | 0.5927 / 0.5962 |
| TL / TL2 | `cardiffnlp/twitter-roberta-large-2022-154m` | big RoBERTa that also learned from 154 million tweets | 0.6114 / 0.6139 |

("What it is" comes from each model's page on Hugging Face.)

**My 2 final submissions:**
- **sub12** = r + t + avg(L, L2) + TL: OOF 0.6220, Public **0.6170** (my best Public)
- **sub14** = r + t + avg(L, L2) + avg(TL, TL2): OOF **0.6268** (my best OOF), Public 0.6111

I dropped distilroberta because it was the weakest, and the ensemble without it was better in 4 of 5 folds (sub9 vs sub8).

**How I decided what to keep.**
- Fine-tuned distilroberta scored 0.6166 on my 80/20 split. On Public it got 0.5776 (trained on all train posts, sub2) and 0.5637 (the same 80% model, sub3).
  Later, its 5-fold OOF score was 0.5784, lower than 0.6166. So one split gave a too-high score, and I changed to 5-fold OOF scores.
- I compared every new idea with my current best on the same 5 folds. In 3.3, I fixed the rule before each run: upload a new idea only if it beats my current best in **4 or 5 of the 5 folds**.
- Ideas that won 4 or 5 of 5 folds: adding roberta-base (5/5), twitter-roberta (4/5), roberta-large (5/5), dropping distilroberta (4/5), new seeds for roberta-large (4/5), twitter-roberta-large (4/5), new seeds for twitter-roberta-large (4/5).
- Ideas I did not use: weights for each emotion tuned on OOF (0/5), deberta-v3 (broke), 4 epochs instead of 3 (2/5), new seeds for twitter-roberta (3/5), counting twitter-roberta-large two times (2/5), mpnet features (gain smaller than how much the score moves from fold to fold).

**My main uploads, step by step (Public score):**

| | sub1 | sub2 | sub5 | sub6 | sub7 | sub11 | sub12 |
|---|---|---|---|---|---|---|---|
| What | TF-IDF + MiniLM + SVM | 1 fine-tuned distilroberta | distilroberta, 5 folds | + roberta-base | + twitter-roberta | r + t + avg(2 roberta-large) | + twitter-roberta-large |
| Public | 0.5321 | 0.5776 | 0.5978 | 0.6057 | 0.6108 | 0.6088 | **0.6170** |

The full list of everything I tried is in the "Experiment & submission log" at the end of this notebook.

**Private leaderboard:** rank 35 of 103 teams, score 0.6435, from sub14 (see the screenshot at the top of this notebook).

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

**AI tool:** Claude (Anthropic), in a chat, from 11/10 to 11/22. I did not use any other AI tool for the competition. I wrote my prompts in Chinese. Below, I rewrote each one in English to make it clearer, but the meaning is the same.
(I also used the course agent in the AgentPipeline notebook, but only for Part 2, not for the competition.)

| # | My prompt (rewritten more clearly) | What the AI suggested | What I decided, and why |
|---|---|---|---|
| 1 | Before I build a model, how should I organize the 4 files and check the data? Which model from the Master is a fair starting point to compare against? | Put the files in `data/kaggle/`, load and merge them, and start from the Master's best model (TF-IDF + SVM) | **Used.** This became sections 1 and 2. |
| 2 | I learned in Exercise 14 that choosing settings by looking at the test score makes the score look too good. How can I check my ideas fairly without the leaderboard, and what kind of features could capture emotion better than word counts? | Keep part of the train data for my own checks; change one thing at a time; try MiniLM sentence embeddings and fine-tuning | **Used.** TF-IDF + MiniLM gave sub1 (Public 0.5321). Fine-tuning gave 0.6166 vs 0.5375 on the same split. |
| 3 | Why is fine-tuning so much slower than an SVM on my laptop, and how can I run it on a GPU? | Run fine-tuning on Kaggle with a free GPU (T4) | **Used.** One epoch took about 6 to 8 minutes on my laptop, and about 17 to 35 seconds on Kaggle for the base models. |
| 4 | My score on one 80/20 split (0.6166) was much higher than Public (0.5776). Why can one split give such a different answer, and how can I measure my model in a way I can trust? | One split can be lucky; use 5-fold out-of-fold scores, and compare ideas fold by fold | **Used.** After this, I compared every idea with 5-fold OOF scores, fold by fold. |
| 5 | Fear has far fewer posts than joy. If I give each emotion a different weight, would the model handle small classes better? How do I make sure the weights do not just fit my own check data? | Tune one weight per emotion on the OOF probabilities, and check it on folds it did not see | **Not used.** It got worse in all 5 folds (0/5), so the weights only fit the data they were tuned on. |
| 6 | Why can training break and give loss `nan`, and what makes training a big model safer? | Use a gentler setting for big models (lr 1e-5, warmup, clipping) and test fold 1 first | **Used** for roberta-large and twitter-roberta-large. deberta-v3 still broke (`nan`), so I dropped it. |
| 7 | (11/16) Two models with the same settings agreed on only 74.9% of test posts. Would training the same model again with other seeds, and averaging, make the answers more stable? How do I test that fairly? | Train the same model again with other seeds and average the two | **Partly used**, only what passed my rule: other seeds for roberta-large (4/5). Not used: other seeds for twitter-roberta (3/5). |
| 8 | (11/17) What kind of model might understand tweets better? And how should I decide, before running, whether a new idea really helps and is not just luck? | Try twitter-roberta-large; count the best model two times; train it again with other seeds | **Partly used:** twitter-roberta-large (4/5) and its other seeds (4/5). Not used: counting it two times (2/5). |
| 9 | My best OOF and my best Public are different submissions. What does each score tell me, and how should I choose when I cannot tell which one is really better?  | One with the best OOF and one with the best Public, because Public is only part of the test data | **Used:** sub14 (best OOF) and sub12 (best Public). |
| 10 | Some Kaggle outputs were not saved, and training again could give slightly different numbers. Is there an honest way to show the same results without training again? | Compute only the scores again, from the saved probability files, on my laptop (3.4) | **Used.** All the scores came out the same as on Kaggle. |
| 11 | Can you teach me step by step how each part of my code works, and help me put all my Kaggle code and results into one clear notebook, so that every number can be checked against a real output? | The AI wrote most of the code. It explained each step to me: what the code does, what the output means, and how to check it | **Used.** I learned how each step works, ran all the code myself, checked the outputs, and made every upload decision. |

---

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [1]:
# ===== 1. Preprocessing: load the files and take a look =====
import json
import pandas as pd

DATA_DIR = "data/kaggle/"   # I put the 4 Kaggle files here

# the posts are in json, so first I just peek at how it looks
with open(DATA_DIR + "final_posts.json", encoding="utf-8") as f:
    raw = json.load(f)

print(type(raw), len(raw))
print(raw[0] if isinstance(raw, list) else list(raw.items())[:1])

# the 3 csv files
emo = pd.read_csv(DATA_DIR + "emotion.csv")
ident = pd.read_csv(DATA_DIR + "data_identification.csv")
sample = pd.read_csv(DATA_DIR + "sampleSubmission.csv")

for name, d in [("emotion", emo), ("identification", ident), ("sample", sample)]:
    print("\n", name, d.shape)
    print(d.head(3))

<class 'list'> 4800
{'post_id': 'f8bv5zh', 'text': 'Thought it looked weird as well.'}

 emotion (3360, 2)
   post_id    label
0  f8bv5zh  disgust
1  q0rr1l9  disgust
2  p94rwgk  sadness

 identification (4800, 2)
   post_id    set
0  f8bv5zh  train
1  q0rr1l9  train
2  p94rwgk  train

 sample (1440, 2)
   post_id    label
0  xyzx9jp  sadness
1  l3vzzxt  sadness
2  g680iug  sadness


In [2]:
# ===== 1. Preprocessing: put the 3 files together =====
posts = pd.DataFrame(raw)                       # 4800 rows: post_id, text
df = posts.merge(ident, on="post_id")           # add train / test

train_df = df[df["set"] == "train"].merge(emo, on="post_id")   # add the label
test_df = df[df["set"] == "test"].reset_index(drop=True)

print("train:", train_df.shape, " test:", test_df.shape)
print("empty texts:", (df["text"].str.strip() == "").sum())

print("\nlabels in train:")
print(train_df["label"].value_counts())

print("\nwords per post:")
print(df["text"].str.split().str.len().describe().round(1))

print("\nsome train posts:")
for t, y in train_df[["text", "label"]].head(8).values:
    print(f"  [{y}] {t[:100]}")

train: (3360, 4)  test: (1440, 3)
empty texts: 0

labels in train:
label
joy         875
anger       674
sadness     628
disgust     454
surprise    441
fear        288
Name: count, dtype: int64

words per post:
count    4800.0
mean       16.3
std        24.9
min         1.0
25%         8.0
50%        13.0
75%        19.0
max       510.0
Name: text, dtype: float64

some train posts:
  [disgust] Thought it looked weird as well.
  [disgust] Bear died in a pit
  [sadness] I have endured worse.
  [fear] The pottery around Brasov is in danger
  [fear] Meanwhile, our soldiers were approaching the area, and the Germans, of course, knew about it and beg
  [disgust] Takes a lot less effort to just say “fuck off, buy your own.”
  [disgust] What’s worse is that the judge and businessman are old college buds.
  [joy] Some have now returned to normal life and are joining forces as national activists in a united commu


## 2. Feature Engineering Steps

In [3]:
# ===== 2. Feature engineering: TF-IDF (same idea as Master 3.1) =====
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# keep 20% of train as my own validation set,
# so I check ideas here instead of on the Kaggle leaderboard (lesson from Ex 14)
tr, va = train_test_split(train_df, test_size=0.2,
                          stratify=train_df["label"], random_state=42)

tfidf = TfidfVectorizer(min_df=2)          # same min_df as my Master
X_tr = tfidf.fit_transform(tr["text"])     # learn the words only from my train part
X_va = tfidf.transform(va["text"])
print("train part:", X_tr.shape, " valid part:", X_va.shape)

train part: (2688, 2548)  valid part: (672, 2548)


In [7]:
# ===== 2. Feature engineering: MiniLM sentence embeddings (same model as Master) =====
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer("all-MiniLM-L6-v2")

# each post becomes 384 numbers that describe its meaning
E_train = st_model.encode(train_df["text"].tolist(), batch_size=64,
                          show_progress_bar=True, normalize_embeddings=True)
E_test = st_model.encode(test_df["text"].tolist(), batch_size=64,
                         show_progress_bar=True, normalize_embeddings=True)
print("train:", E_train.shape, " test:", E_test.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/53 [00:00<?, ?it/s]

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

train: (3360, 384)  test: (1440, 384)


In [13]:
# ===== 2. Feature engineering: a stronger sentence model, all-mpnet-base-v2 =====
mp_model = SentenceTransformer("all-mpnet-base-v2")

# each post becomes 768 numbers (MiniLM gave 384)
M_train = mp_model.encode(train_df["text"].tolist(), batch_size=32,
                          show_progress_bar=True, normalize_embeddings=True)
M_test = mp_model.encode(test_df["text"].tolist(), batch_size=32,
                         show_progress_bar=True, normalize_embeddings=True)
print("train:", M_train.shape, " test:", M_test.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

... (6 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/105 [00:00<?, ?it/s]

Batches:   0%|          | 0/45 [00:00<?, ?it/s]

train: (3360, 768)  test: (1440, 768)


## 3. Model Implementation Steps

### 3.1 First tries on my laptop (CPU)

All cells in 3.1 ran here, on my laptop.
- I started with the best model from the Master: TF-IDF + Linear SVM.
- I changed **one thing at a time**, so I knew what made the score go up or down.
- At first I tested on one 80/20 split. The scores jumped around too much, so I changed to **5-fold cross-validation** (CV).
- Then I **fine-tuned** a small transformer model (distilroberta). On the same split it was much better: 0.6166 vs 0.5375.
- But one epoch took about 6 minutes on my laptop. So I trained the final models (3.2 and 3.3) on Kaggle, which has a GPU.

In [4]:
# ===== 3. Model: Linear SVM baseline (my best model in the Master) =====
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report

svm = LinearSVC(C=1.0)
svm.fit(X_tr, tr["label"])
pred_va = svm.predict(X_va)

print("valid macro F1:", round(f1_score(va["label"], pred_va, average="macro"), 4))
print(classification_report(va["label"], pred_va, digits=3))

valid macro F1: 0.4806
              precision    recall  f1-score   support

       anger      0.439     0.452     0.445       135
     disgust      0.357     0.275     0.311        91
        fear      0.568     0.368     0.447        57
         joy      0.584     0.657     0.618       175
     sadness      0.557     0.579     0.568       126
    surprise      0.469     0.523     0.495        88

    accuracy                          0.507       672
   macro avg      0.496     0.476     0.481       672
weighted avg      0.503     0.507     0.502       672



In [5]:
# ===== 3. Model: try one change at a time, everything else stays the same =====
results = []

def try_it(name, vec, model):
    Xa = vec.fit_transform(tr["text"])      # words learned only from my train part
    Xb = vec.transform(va["text"])
    model.fit(Xa, tr["label"])
    f1 = f1_score(va["label"], model.predict(Xb), average="macro")
    results.append({"try": name, "valid_macro_f1": round(f1, 4)})

try_it("baseline: min_df=2, C=1",   TfidfVectorizer(min_df=2), LinearSVC(C=1.0))
try_it("A: min_df=1",               TfidfVectorizer(min_df=1), LinearSVC(C=1.0))
try_it("B: class_weight=balanced",  TfidfVectorizer(min_df=2), LinearSVC(C=1.0, class_weight="balanced"))
try_it("C: add 2-word phrases",     TfidfVectorizer(min_df=2, ngram_range=(1, 2)), LinearSVC(C=1.0))

pd.DataFrame(results)

,try,valid_macro_f1
0,"baseline: min_df=2, C=1",0.4806
1,A: min_df=1,0.5099
2,B: class_weight=balanced,0.4870
3,C: add 2-word phrases,0.4898


In [8]:
# ===== 3. Model: 5-fold cross-validation, more stable than one split =====
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

def cv_it(name, vec, model):
    pipe = make_pipeline(vec, model)   # TF-IDF is learned again inside each fold, so no peeking
    s = cross_val_score(pipe, train_df["text"], train_df["label"], cv=cv, scoring="f1_macro")
    cv_results.append({"try": name, "cv_mean": round(s.mean(), 4), "cv_std": round(s.std(), 4)})

cv_it("baseline: min_df=2",   TfidfVectorizer(min_df=2), LinearSVC(C=1.0))
cv_it("A: min_df=1",          TfidfVectorizer(min_df=1), LinearSVC(C=1.0))
cv_it("A + B (balanced)",     TfidfVectorizer(min_df=1), LinearSVC(C=1.0, class_weight="balanced"))
cv_it("A + C (2-word)",       TfidfVectorizer(min_df=1, ngram_range=(1, 2)), LinearSVC(C=1.0))
cv_it("A + B + C",            TfidfVectorizer(min_df=1, ngram_range=(1, 2)), LinearSVC(C=1.0, class_weight="balanced"))

pd.DataFrame(cv_results)

,try,cv_mean,cv_std
0,baseline: min_df=2,0.4625,0.0229
1,A: min_df=1,0.4768,0.0143
2,A + B (balanced),0.4846,0.0199
3,A + C (2-word),0.4860,0.0063
4,A + B + C,0.4842,0.0056


In [9]:
# ===== 3. Model: MiniLM features + Linear SVM, same 5-fold CV =====
def cv_dense(name, X, model):
    s = cross_val_score(model, X, train_df["label"], cv=cv, scoring="f1_macro")
    cv_results.append({"try": name, "cv_mean": round(s.mean(), 4), "cv_std": round(s.std(), 4)})

cv_dense("MiniLM + SVM",           E_train, LinearSVC(C=1.0))
cv_dense("MiniLM + SVM balanced",  E_train, LinearSVC(C=1.0, class_weight="balanced"))

pd.DataFrame(cv_results)

,try,cv_mean,cv_std
0,baseline: min_df=2,0.4625,0.0229
1,A: min_df=1,0.4768,0.0143
2,A + B (balanced),0.4846,0.0199
3,A + C (2-word),0.4860,0.0063
4,A + B + C,0.4842,0.0056
5,MiniLM + SVM,0.5003,0.0243
6,MiniLM + SVM balanced,0.4936,0.0258


In [14]:
# ===== 3. Model: TF-IDF (A + C) and MiniLM side by side, same 5-fold CV =====
# (row 8 shows up twice in the output because I ran this cell twice)
import numpy as np
from scipy.sparse import hstack, csr_matrix

def cv_combo(name):
    scores = []
    for tr_i, va_i in cv.split(train_df["text"], train_df["label"]):
        vec = TfidfVectorizer(min_df=1, ngram_range=(1, 2))   # learn words only from this fold's train
        Xa = hstack([vec.fit_transform(train_df["text"].iloc[tr_i]), csr_matrix(E_train[tr_i])])
        Xb = hstack([vec.transform(train_df["text"].iloc[va_i]),     csr_matrix(E_train[va_i])])
        m = LinearSVC(C=1.0).fit(Xa, train_df["label"].iloc[tr_i])
        scores.append(f1_score(train_df["label"].iloc[va_i], m.predict(Xb), average="macro"))
    cv_results.append({"try": name, "cv_mean": round(np.mean(scores), 4), "cv_std": round(np.std(scores), 4)})

cv_combo("TF-IDF (A+C) + MiniLM")
pd.DataFrame(cv_results)

,try,cv_mean,cv_std
0,baseline: min_df=2,0.4625,0.0229
1,A: min_df=1,0.4768,0.0143
2,A + B (balanced),0.4846,0.0199
3,A + C (2-word),0.4860,0.0063
4,A + B + C,0.4842,0.0056
5,MiniLM + SVM,0.5003,0.0243
6,MiniLM + SVM balanced,0.4936,0.0258
7,TF-IDF (A+C) + MiniLM,0.5313,0.0206
8,TF-IDF (A+C) + MiniLM,0.5313,0.0206


In [15]:
# ===== 3. Model: train the best one (TF-IDF A+C + MiniLM) on ALL train, then predict test =====
import os

vec_final = TfidfVectorizer(min_df=1, ngram_range=(1, 2))
X_all  = hstack([vec_final.fit_transform(train_df["text"]), csr_matrix(E_train)])
X_test = hstack([vec_final.transform(test_df["text"]),      csr_matrix(E_test)])

final_svm = LinearSVC(C=1.0).fit(X_all, train_df["label"])
test_df["label"] = final_svm.predict(X_test)

# follow sampleSubmission.csv: same ids, same order, columns post_id and label
sub = sample[["post_id"]].merge(test_df[["post_id", "label"]], on="post_id", how="left")
print(sub.shape, " missing labels:", sub["label"].isna().sum())
print(sub["label"].value_counts())

os.makedirs("submissions", exist_ok=True)
sub.to_csv("submissions/sub1_tfidf_minilm_svm.csv", index=False)
print(sub.head())

(1440, 2)  missing labels: 0
label
joy         441
anger       331
sadness     263
disgust     159
surprise    151
fear         95
Name: count, dtype: int64
   post_id     label
0  xyzx9jp     anger
1  l3vzzxt   sadness
2  g680iug   sadness
3  m4t8bp6       joy
4  ai3al6e  surprise


In [12]:
# Kaggle said "ID column id not found", and the Overview asks for "id", so I rename the column
sub_id = sub.rename(columns={"post_id": "id"})
sub_id.to_csv("submissions/sub1_tfidf_minilm_svm_id.csv", index=False)
print(sub_id.shape)
print(sub_id.head(3))

(1440, 2)
        id    label
0  xyzx9jp    anger
1  l3vzzxt  sadness
2  g680iug  sadness


In [16]:
# ===== 3. Model: TF-IDF (A+C) + a sentence model, same 5-fold CV =====
def cv_combo2(name, E):
    scores = []
    for tr_i, va_i in cv.split(train_df["text"], train_df["label"]):
        vec = TfidfVectorizer(min_df=1, ngram_range=(1, 2))
        Xa = hstack([vec.fit_transform(train_df["text"].iloc[tr_i]), csr_matrix(E[tr_i])])
        Xb = hstack([vec.transform(train_df["text"].iloc[va_i]),     csr_matrix(E[va_i])])
        m = LinearSVC(C=1.0).fit(Xa, train_df["label"].iloc[tr_i])
        scores.append(f1_score(train_df["label"].iloc[va_i], m.predict(Xb), average="macro"))
    cv_results.append({"try": name, "cv_mean": round(np.mean(scores), 4), "cv_std": round(np.std(scores), 4)})

cv_dense("mpnet + SVM", M_train, LinearSVC(C=1.0))
cv_combo2("TF-IDF (A+C) + mpnet", M_train)
cv_combo2("TF-IDF (A+C) + MiniLM + mpnet", np.hstack([E_train, M_train]))

pd.DataFrame(cv_results)

,try,cv_mean,cv_std
0,baseline: min_df=2,0.4625,0.0229
1,A: min_df=1,0.4768,0.0143
2,A + B (balanced),0.4846,0.0199
3,A + C (2-word),0.4860,0.0063
4,A + B + C,0.4842,0.0056
5,MiniLM + SVM,0.5003,0.0243
6,MiniLM + SVM balanced,0.4936,0.0258
7,TF-IDF (A+C) + MiniLM,0.5313,0.0206
8,TF-IDF (A+C) + MiniLM,0.5313,0.0206
9,mpnet + SVM,0.4806,0.0304


In [18]:
# ===== 3. Model: fine-tune distilroberta, the whole model learns our 6 emotions =====
import time, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

torch.manual_seed(42)
np.random.seed(42)

LABELS = sorted(train_df["label"].unique())            # the 6 emotions
lab2id = {l: i for i, l in enumerate(LABELS)}
MODEL_NAME = "distilroberta-base"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def batches(texts, labels=None, bs=16, shuffle=False):
    # cut the data into small batches, each batch is padded only to its own longest post
    idx = np.random.permutation(len(texts)) if shuffle else np.arange(len(texts))
    for s in range(0, len(idx), bs):
        b = idx[s:s + bs]
        enc = tok([texts[i] for i in b], truncation=True, max_length=128,
                  padding=True, return_tensors="pt")
        y = None if labels is None else torch.tensor([labels[i] for i in b])
        yield enc, y

def train_model(texts, labels, epochs=3, lr=3e-5):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(LABELS))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    model.train()
    for ep in range(epochs):
        t0, total = time.time(), 0.0
        for enc, y in batches(texts, labels, shuffle=True):
            out = model(**enc, labels=y)
            out.loss.backward()
            opt.step()
            opt.zero_grad()
            total += out.loss.item()
        print(f"epoch {ep + 1}: total loss {total:.1f}, {time.time() - t0:.0f}s")
    return model

def predict(model, texts):
    model.eval()
    preds = []
    with torch.no_grad():
        for enc, _ in batches(texts, bs=64):
            preds += model(**enc).logits.argmax(-1).tolist()
    return [LABELS[p] for p in preds]

# train on my 80% part, check on the same 20% part as before
ft_model = train_model(tr["text"].tolist(), [lab2id[l] for l in tr["label"]], epochs=3)
pred_ft = predict(ft_model, va["text"].tolist())
print("\nfine-tuned distilroberta, holdout macro F1:",
      round(f1_score(va["label"], pred_ft, average="macro"), 4))

# fair comparison: my sub1 method (TF-IDF A+C + MiniLM) on the SAME 20% part
vec_h = TfidfVectorizer(min_df=1, ngram_range=(1, 2))
Xa = hstack([vec_h.fit_transform(tr["text"]), csr_matrix(E_train[tr.index])])
Xb = hstack([vec_h.transform(va["text"]),     csr_matrix(E_train[va.index])])
pred_sub1 = LinearSVC(C=1.0).fit(Xa, tr["label"]).predict(Xb)
print("sub1 method (TF-IDF + MiniLM), holdout macro F1:",
      round(f1_score(va["label"], pred_sub1, average="macro"), 4))

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  331MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | Details
----------------------------+------------+--------
roberta.pooler.dense.weight | UNEXPECTED |        
lm_head.layer_norm.bias     | UNEXPECTED |        
lm_head.dense.bias          | UNEXPECTED |        
lm_head.bias                | UNEXPECTED |        
roberta.pooler.dense.bias   | UNEXPECTED |        
lm_head.dense.weight        | UNEXPECTED |        
lm_head.layer_norm.weight   | UNEXPECTED |        
classifier.out_proj.weight  | MISSING    |        
classifier.dense.weight     | MISSING    |        
classifier.dense.bias       | MISSING    |        
classifier.out_proj.bias    | MISSING    |        

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1: total loss 245.4, 365s
epoch 2: total loss 168.7, 351s
epoch 3: total loss 121.1, 356s

fine-tuned distilroberta, holdout macro F1: 0.6166
sub1 method (TF-IDF + MiniLM), holdout macro F1: 0.5375


In [19]:
# ===== 3. Model: fine-tune distilroberta on ALL train, then predict test =====
torch.manual_seed(42)
np.random.seed(42)

ft_all = train_model(train_df["text"].tolist(), [lab2id[l] for l in train_df["label"]], epochs=3)
test_df["label"] = predict(ft_all, test_df["text"].tolist())

# same ids and order as sampleSubmission, and the column must be called "id"
sub2 = sample[["post_id"]].merge(test_df[["post_id", "label"]], on="post_id", how="left")
sub2 = sub2.rename(columns={"post_id": "id"})
print(sub2.shape, " missing labels:", sub2["label"].isna().sum())
print(sub2["label"].value_counts())

sub2.to_csv("submissions/sub2_distilroberta_ft.csv", index=False)
print(sub2.head(3))

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | Details
----------------------------+------------+--------
roberta.pooler.dense.weight | UNEXPECTED |        
lm_head.layer_norm.bias     | UNEXPECTED |        
lm_head.dense.bias          | UNEXPECTED |        
lm_head.bias                | UNEXPECTED |        
roberta.pooler.dense.bias   | UNEXPECTED |        
lm_head.dense.weight        | UNEXPECTED |        
lm_head.layer_norm.weight   | UNEXPECTED |        
classifier.out_proj.weight  | MISSING    |        
classifier.dense.weight     | MISSING    |        
classifier.dense.bias       | MISSING    |        
classifier.out_proj.bias    | MISSING    |        

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1: total loss 293.8, 468s
epoch 2: total loss 202.4, 460s
epoch 3: total loss 149.7, 458s
(1440, 2)  missing labels: 0
label
joy         414
anger       326
sadness     265
surprise    227
fear        139
disgust      69
Name: count, dtype: int64
        id     label
0  xyzx9jp  surprise
1  l3vzzxt   sadness
2  g680iug   sadness


In [20]:
# ===== 3. Model: why is Public (0.5776) lower than my holdout (0.6166)? =====
# look at the holdout predictions of the fine-tuned model, class by class
print(classification_report(va["label"], pred_ft, digits=3))

print("predicted on holdout (672 posts):")
print(pd.Series(pred_ft).value_counts())
print("\npredicted on test (1440 posts):")
print(test_df["label"].value_counts())

              precision    recall  f1-score   support

       anger      0.570     0.726     0.638       135
     disgust      0.406     0.429     0.417        91
        fear      0.633     0.544     0.585        57
         joy      0.747     0.811     0.778       175
     sadness      0.826     0.603     0.697       126
    surprise      0.644     0.534     0.584        88

    accuracy                          0.644       672
   macro avg      0.638     0.608     0.617       672
weighted avg      0.657     0.644     0.644       672

predicted on holdout (672 posts):
joy         190
anger       172
disgust      96
sadness      92
surprise     73
fear         49
Name: count, dtype: int64

predicted on test (1440 posts):
label
joy         414
anger       326
sadness     265
surprise    227
fear        139
disgust      69
Name: count, dtype: int64


In [21]:
# ===== 3. Model: is the problem the test posts, or the retrained model? =====
pred_test_holdout_model = predict(ft_model, test_df["text"].tolist())
print("holdout model, predicted on test:")
print(pd.Series(pred_test_holdout_model).value_counts())

# how often do the two fine-tuned models agree on the test posts?
agree = (pd.Series(pred_test_holdout_model) == test_df["label"]).mean()
print("\nthe two models agree on", round(agree * 100, 1), "% of test posts")

holdout model, predicted on test:
anger       407
joy         366
disgust     212
sadness     193
surprise    170
fear         92
Name: count, dtype: int64

the two models agree on 74.9 % of test posts


In [22]:
# ===== 3. Model: average the two fine-tuned models (ensemble) =====
def predict_proba(model, texts):
    # the model's confidence for each of the 6 emotions (each row adds up to 1)
    model.eval()
    out = []
    with torch.no_grad():
        for enc, _ in batches(texts, bs=64):
            out.append(torch.softmax(model(**enc).logits, dim=-1).numpy())
    return np.vstack(out)

test_texts = test_df["text"].tolist()
P_hold = predict_proba(ft_model, test_texts)   # model trained on 80%
P_all  = predict_proba(ft_all,  test_texts)    # model trained on 100%

pred_hold = [LABELS[i] for i in P_hold.argmax(axis=1)]
pred_ens  = [LABELS[i] for i in (P_hold + P_all).argmax(axis=1)]

def save_sub(preds, path):
    tmp = pd.DataFrame({"post_id": test_df["post_id"], "label": preds})
    s = sample[["post_id"]].merge(tmp, on="post_id", how="left").rename(columns={"post_id": "id"})
    print(path, s.shape, " missing:", s["label"].isna().sum())
    print(s["label"].value_counts(), "\n")
    s.to_csv(path, index=False)

save_sub(pred_hold, "submissions/sub3_ft_holdout_model.csv")
save_sub(pred_ens,  "submissions/sub4_ft_ensemble2.csv")

submissions/sub3_ft_holdout_model.csv (1440, 2)  missing: 0
label
anger       407
joy         366
disgust     212
sadness     193
surprise    170
fear         92
Name: count, dtype: int64 

submissions/sub4_ft_ensemble2.csv (1440, 2)  missing: 0
label
joy         392
anger       376
sadness     235
surprise    199
fear        121
disgust     117
Name: count, dtype: int64 



In [23]:
# ===== 3. Model: tune one weight per emotion on out-of-fold probabilities (no retraining) =====
from sklearn.model_selection import StratifiedKFold

LABELS = sorted(train_df["label"].unique())
y_all = np.array([LABELS.index(l) for l in train_df["label"]])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)   # same folds as on Kaggle
folds = list(skf.split(train_df["text"], y_all))

# probabilities from the Kaggle GPU runs (distilroberta + roberta-base, 5-fold each)
P_oof = np.load("submissions/oof_distilroberta.npy") + np.load("submissions/oof_roberta_base.npy")
P_tst = np.load("submissions/test_distilroberta.npy") + np.load("submissions/test_roberta_base.npy")
print("check, should be 0.5983:", round(f1_score(y_all, P_oof.argmax(axis=1), average="macro"), 4))

def tune_weights(P, y, grid=np.arange(0.5, 2.01, 0.1), rounds=3):
    # try each emotion's weight one at a time, keep the value with the best macro F1
    w = np.ones(P.shape[1])
    for _ in range(rounds):
        for c in range(P.shape[1]):
            scores = []
            for v in grid:
                w2 = w.copy()
                w2[c] = v
                scores.append(f1_score(y, (P * w2).argmax(axis=1), average="macro"))
            w[c] = grid[int(np.argmax(scores))]
    return w

# fair check: weights learned on the other 4 folds, scored on the fold they never saw
wins = 0
for k, (tr_i, va_i) in enumerate(folds):
    w = tune_weights(P_oof[tr_i], y_all[tr_i])
    before = f1_score(y_all[va_i], P_oof[va_i].argmax(axis=1), average="macro")
    after = f1_score(y_all[va_i], (P_oof[va_i] * w).argmax(axis=1), average="macro")
    wins += after > before
    print(f"fold {k + 1}: before {before:.4f}  after {after:.4f}  weights {np.round(w, 1)}")
print(f"\nweights help in {wins} of 5 folds")

check, should be 0.5983: 0.5983
fold 1: before 0.5836  after 0.5786  weights [0.9 1.  0.7 1.1 1.  0.9]
fold 2: before 0.5584  after 0.5497  weights [1.4 2.  1.  1.2 1.1 1. ]
fold 3: before 0.6334  after 0.6130  weights [1.1 1.1 0.8 1.7 1.5 0.7]
fold 4: before 0.6123  after 0.6116  weights [0.9 1.  0.7 1.  0.9 1.2]
fold 5: before 0.6046  after 0.6032  weights [1.  1.  0.7 1.4 1.2 0.9]

weights help in 0 of 5 folds


### 3.2 Final models, part 1 (ran on Kaggle GPU)

I ran these cells on **Kaggle** (GPU: Tesla T4), in my Kaggle notebook `dmLab2_try01`.
The outputs are from that Kaggle run. I only removed very long messages (model loading reports and progress bars).

**Please do not run 3.2 or 3.3 here.** They need a GPU, and running them would replace the Kaggle outputs.
The numbers next to these cells (like [9]) are run numbers from Kaggle, not from this notebook. Cells with an empty [ ] did not run in this notebook either.

**Words I use:**
- **fold**: I cut the 3,360 train posts into 5 parts. Each time, the model learns from 4 parts and is tested on the 5th part. I do this 5 times.
- **OOF score** (out-of-fold): put the 5 test parts together, so every train post gets a prediction from a model that never saw it. Then I compute macro F1 on all 3,360 posts.
- I use the **same 5 folds** everywhere (`StratifiedKFold(5, shuffle=True, random_state=42)`), so I can compare two ideas fold by fold.
- **ensemble**: I add up the probabilities of several models and pick the emotion with the highest total.

**Short names:** **d** = distilroberta-base, **r** = roberta-base, **t** = twitter-roberta-base, **L** = roberta-large.

In [9]:
# ===== 1. Load data (same steps as my Homework notebook) =====
import os, json, time
import numpy as np
import pandas as pd
import torch

# find the competition folder automatically
DATA_DIR = [r for r, d, f in os.walk("/kaggle/input") if "final_posts.json" in f][0] + "/"
print("data folder:", DATA_DIR)

with open(DATA_DIR + "final_posts.json", encoding="utf-8") as f:
    raw = json.load(f)
emo = pd.read_csv(DATA_DIR + "emotion.csv")
ident = pd.read_csv(DATA_DIR + "data_identification.csv")
sample = pd.read_csv(DATA_DIR + "sampleSubmission.csv")

posts = pd.DataFrame(raw)
df = posts.merge(ident, on="post_id")
train_df = df[df["set"] == "train"].merge(emo, on="post_id")
test_df = df[df["set"] == "test"].reset_index(drop=True)
print("train:", train_df.shape, " test:", test_df.shape)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

data folder: /kaggle/input/competitions/2026-dm-lab-2-private-competition/
train: (3360, 4)  test: (1440, 3)
device: cuda
GPU: Tesla T4


In [11]:
# ===== 2. Fine-tuning helpers =====
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report

LABELS = sorted(train_df["label"].unique())          # the 6 emotions
lab2id = {l: i for i, l in enumerate(LABELS)}

texts = train_df["text"].tolist()
y_all = np.array([lab2id[l] for l in train_df["label"]])
test_texts = test_df["text"].tolist()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)   # same folds as my Homework CV

tok, MODEL_NAME = None, None                           # set inside run_5fold()

def batches(texts, labels=None, bs=16, shuffle=False):
    # small batches, each padded only to its own longest post, then moved to the GPU
    idx = np.random.permutation(len(texts)) if shuffle else np.arange(len(texts))
    for s in range(0, len(idx), bs):
        b = idx[s:s + bs]
        enc = tok([texts[i] for i in b], truncation=True, max_length=128,
                  padding=True, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        y = None if labels is None else torch.tensor([labels[i] for i in b]).to(device)
        yield enc, y

def train_model(texts, labels, epochs=3, lr=3e-5, seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABELS)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    model.train()
    for ep in range(epochs):
        t0, total = time.time(), 0.0
        for enc, y in batches(texts, labels, shuffle=True):
            out = model(**enc, labels=y)
            out.loss.backward()
            opt.step()
            opt.zero_grad()
            total += out.loss.item()
        print(f"  epoch {ep + 1}: total loss {total:.1f}, {time.time() - t0:.0f}s")
    return model

def predict_proba(model, texts):
    # the model's confidence for each of the 6 emotions (each row adds up to 1)
    model.eval()
    out = []
    with torch.no_grad():
        for enc, _ in batches(texts, bs=64):
            out.append(torch.softmax(model(**enc).logits, dim=-1).cpu().numpy())
    return np.vstack(out)

def run_5fold(model_name, epochs=3, lr=3e-5):
    # train 5 models on the same 5 folds; return out-of-fold and averaged test probabilities
    global tok, MODEL_NAME
    MODEL_NAME = model_name
    tok = AutoTokenizer.from_pretrained(model_name)
    oof_m = np.zeros((len(texts), len(LABELS)))
    P_m = np.zeros((len(test_texts), len(LABELS)))
    f1s = []
    for k, (tr_i, va_i) in enumerate(skf.split(texts, y_all)):
        print(f"{model_name} fold {k + 1}")
        m = train_model([texts[i] for i in tr_i], y_all[tr_i].tolist(),
                        epochs=epochs, lr=lr, seed=42 + k)
        oof_m[va_i] = predict_proba(m, [texts[i] for i in va_i])
        P_m += predict_proba(m, test_texts) / 5
        f1s.append(f1_score(y_all[va_i], oof_m[va_i].argmax(axis=1), average="macro"))
        print(f"  fold {k + 1} macro F1: {f1s[-1]:.4f}")
        del m
        torch.cuda.empty_cache()
    print(f"{model_name} 5-fold mean: {np.mean(f1s):.4f}  std: {np.std(f1s):.4f}\n")
    return oof_m, P_m

def save_sub(P, path):
    # follow sampleSubmission: same ids and order, column must be called "id"
    pred = [LABELS[i] for i in P.argmax(axis=1)]
    tmp = pd.DataFrame({"post_id": test_df["post_id"], "label": pred})
    s = sample[["post_id"]].merge(tmp, on="post_id", how="left").rename(columns={"post_id": "id"})
    print(path, s.shape, " missing:", s["label"].isna().sum())
    print(s["label"].value_counts(), "\n")
    s.to_csv(path, index=False)

print("ready, labels:", LABELS)

ready, labels: ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']


In [12]:
# ===== 3. Model A: distilroberta, 5-fold (sub5 = average of the 5 fold models) =====
oof_d, P_d = run_5fold("distilroberta-base")
np.save("/kaggle/working/oof_distilroberta.npy", oof_d)
np.save("/kaggle/working/test_distilroberta.npy", P_d)
print(classification_report(y_all, oof_d.argmax(axis=1), target_names=LABELS, digits=3))
save_sub(P_d, "/kaggle/working/sub5_distilroberta_5fold.csv")

distilroberta-base fold 1
  epoch 1: total loss 241.6, 17s
  epoch 2: total loss 163.8, 18s
  epoch 3: total loss 119.3, 18s
  fold 1 macro F1: 0.5672
distilroberta-base fold 2
  epoch 1: total loss 237.7, 18s
  epoch 2: total loss 162.6, 18s
  epoch 3: total loss 116.5, 18s
  fold 2 macro F1: 0.5462
distilroberta-base fold 3
  epoch 1: total loss 245.8, 18s
  epoch 2: total loss 170.6, 18s
  epoch 3: total loss 125.7, 18s
  fold 3 macro F1: 0.6042
distilroberta-base fold 4
  epoch 1: total loss 237.0, 18s
  epoch 2: total loss 163.9, 18s
  epoch 3: total loss 118.7, 18s
  fold 4 macro F1: 0.5930
distilroberta-base fold 5
  epoch 1: total loss 250.2, 18s
  epoch 2: total loss 174.1, 18s
  epoch 3: total loss 127.0, 18s
  fold 5 macro F1: 0.5853
distilroberta-base 5-fold mean: 0.5792  std: 0.0204
              precision    recall  f1-score   support
       anger      0.538     0.623     0.578       674
     disgust      0.417     0.392     0.404       454
        fear      0.654     0.5

In [13]:
# ===== 4. Model B: roberta-base, same 5 folds =====
oof_r, P_r = run_5fold("roberta-base")
np.save("/kaggle/working/oof_roberta_base.npy", oof_r)
np.save("/kaggle/working/test_roberta_base.npy", P_r)
print(classification_report(y_all, oof_r.argmax(axis=1), target_names=LABELS, digits=3))

roberta-base fold 1
  epoch 1: total loss 237.4, 34s
  epoch 2: total loss 160.6, 35s
  epoch 3: total loss 116.1, 34s
  fold 1 macro F1: 0.5695
roberta-base fold 2
  epoch 1: total loss 253.7, 34s
  epoch 2: total loss 174.2, 33s
  epoch 3: total loss 124.3, 34s
  fold 2 macro F1: 0.5528
roberta-base fold 3
  epoch 1: total loss 245.9, 34s
  epoch 2: total loss 166.4, 34s
  epoch 3: total loss 122.2, 34s
  fold 3 macro F1: 0.6258
roberta-base fold 4
  epoch 1: total loss 240.7, 34s
  epoch 2: total loss 167.7, 34s
  epoch 3: total loss 118.6, 34s
  fold 4 macro F1: 0.5969
roberta-base fold 5
  epoch 1: total loss 249.7, 33s
  epoch 2: total loss 174.8, 33s
  epoch 3: total loss 127.6, 34s
  fold 5 macro F1: 0.5909
roberta-base 5-fold mean: 0.5872  std: 0.0249
              precision    recall  f1-score   support
       anger      0.531     0.623     0.573       674
     disgust      0.426     0.377     0.400       454
        fear      0.668     0.580     0.621       288
         joy 

In [14]:
# ===== 5. Ensemble: average distilroberta + roberta-base (sub6) =====
for name, o in [("distilroberta", oof_d), ("roberta-base", oof_r), ("average of both", oof_d + oof_r)]:
    print(f"{name:16s} OOF macro F1: {f1_score(y_all, o.argmax(axis=1), average='macro'):.4f}")

# paired check on the same folds: does the average beat both single models every time?
wins = 0
for k, (tr_i, va_i) in enumerate(skf.split(texts, y_all)):
    f_d = f1_score(y_all[va_i], oof_d[va_i].argmax(axis=1), average="macro")
    f_r = f1_score(y_all[va_i], oof_r[va_i].argmax(axis=1), average="macro")
    f_a = f1_score(y_all[va_i], (oof_d + oof_r)[va_i].argmax(axis=1), average="macro")
    wins += f_a > max(f_d, f_r)
    print(f"fold {k + 1}: distil {f_d:.4f}  roberta {f_r:.4f}  average {f_a:.4f}")
print(f"\nthe average beats both single models in {wins} of 5 folds\n")

save_sub(P_d + P_r, "/kaggle/working/sub6_distil_roberta_avg.csv")

distilroberta    OOF macro F1: 0.5784
roberta-base     OOF macro F1: 0.5879
average of both  OOF macro F1: 0.5983
fold 1: distil 0.5672  roberta 0.5695  average 0.5836
fold 2: distil 0.5462  roberta 0.5528  average 0.5584
fold 3: distil 0.6042  roberta 0.6258  average 0.6334
fold 4: distil 0.5930  roberta 0.5969  average 0.6123
fold 5: distil 0.5853  roberta 0.5909  average 0.6046
the average beats both single models in 5 of 5 folds
/kaggle/working/sub6_distil_roberta_avg.csv (1440, 2)  missing: 0
label
anger       368
joy         356
sadness     295
disgust     171
surprise    156
fear         94
Name: count, dtype: int64 


#### Extra tests on Kaggle
- **deberta-v3-base** did not work. The loss was `nan` (not a number) from the first epoch, even with a gentler setting, and it said "anger" for every post. So I stopped using it.
- **twitter-roberta**, fold 1 only: a quick test to make sure training works, before I spend time on all 5 folds.

In [15]:
# ===== test deberta-v3 on fold 1 only, with a gentler training recipe =====
from transformers import get_linear_schedule_with_warmup

def train_model_safe(texts, labels, epochs=3, lr=2e-5, seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABELS)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = epochs * int(np.ceil(len(texts) / 16))
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)   # warm up, then slow down
    model.train()
    for ep in range(epochs):
        t0, total = time.time(), 0.0
        for enc, y in batches(texts, labels, shuffle=True):
            out = model(**enc, labels=y)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)          # stop huge updates
            opt.step()
            sched.step()
            opt.zero_grad()
            total += out.loss.item()
        print(f"  epoch {ep + 1}: total loss {total:.1f}, {time.time() - t0:.0f}s")
    return model

MODEL_NAME = "microsoft/deberta-v3-base"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

tr_i, va_i = next(iter(skf.split(texts, y_all)))      # fold 1 only
m = train_model_safe([texts[i] for i in tr_i], y_all[tr_i].tolist())
p = predict_proba(m, [texts[i] for i in va_i])

print("NaN in predictions:", np.isnan(p).any())
print("fold 1 macro F1:", round(f1_score(y_all[va_i], p.argmax(axis=1), average="macro"), 4),
      " (distil 0.5672, roberta 0.5695)")
print(pd.Series([LABELS[i] for i in p.argmax(axis=1)]).value_counts())

  epoch 1: total loss nan, 22s
  epoch 2: total loss nan, 19s
  epoch 3: total loss nan, 19s
NaN in predictions: True
fold 1 macro F1: 0.0558  (distil 0.5672, roberta 0.5695)
anger    672
Name: count, dtype: int64


In [16]:
# ===== test another model on fold 1 only: roberta pre-trained on tweets =====
MODEL_NAME = "cardiffnlp/twitter-roberta-base"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

tr_i, va_i = next(iter(skf.split(texts, y_all)))      # fold 1 only
m = train_model([texts[i] for i in tr_i], y_all[tr_i].tolist(), seed=42)   # same recipe as distil / roberta
p = predict_proba(m, [texts[i] for i in va_i])

print("NaN in predictions:", np.isnan(p).any())
print("fold 1 macro F1:", round(f1_score(y_all[va_i], p.argmax(axis=1), average="macro"), 4),
      " (distil 0.5672, roberta 0.5695)")
print(pd.Series([LABELS[i] for i in p.argmax(axis=1)]).value_counts())
del m
torch.cuda.empty_cache()

  epoch 1: total loss 228.1, 34s
  epoch 2: total loss 143.5, 35s
  epoch 3: total loss 90.9, 34s
NaN in predictions: False
fold 1 macro F1: 0.5588  (distil 0.5672, roberta 0.5695)
joy         160
sadness     129
disgust     124
anger       101
surprise     93
fear         65
Name: count, dtype: int64


Note: my Kaggle session restarted here, so the trained results in memory were lost.
I had saved the probabilities of distilroberta and roberta-base as files in a private Kaggle dataset, so I loaded them back instead of training again.
If you run this part from the top in one go, you do not need this step.

In [20]:
# ===== Model C: twitter-roberta, same 5 folds, then check the 3-model average =====
oof_t, P_t = run_5fold("cardiffnlp/twitter-roberta-base")
np.save("/kaggle/working/oof_twitter_roberta.npy", oof_t)
np.save("/kaggle/working/test_twitter_roberta.npy", P_t)

for name, o in [("twitter-roberta", oof_t),
                ("distil + roberta (sub6)", oof_d + oof_r),
                ("distil + roberta + twitter", oof_d + oof_r + oof_t)]:
    print(f"{name:27s} OOF macro F1: {f1_score(y_all, o.argmax(axis=1), average='macro'):.4f}")

# paired check: does adding twitter-roberta beat sub6's 2-model average in every fold?
wins = 0
for k, (tr_i, va_i) in enumerate(skf.split(texts, y_all)):
    f2 = f1_score(y_all[va_i], (oof_d + oof_r)[va_i].argmax(axis=1), average="macro")
    f3 = f1_score(y_all[va_i], (oof_d + oof_r + oof_t)[va_i].argmax(axis=1), average="macro")
    wins += f3 > f2
    print(f"fold {k + 1}: 2 models {f2:.4f}   3 models {f3:.4f}")
print(f"\n3 models beat 2 models in {wins} of 5 folds\n")

save_sub(P_d + P_r + P_t, "/kaggle/working/sub7_distil_roberta_twitter_avg.csv")

cardiffnlp/twitter-roberta-base fold 1
  epoch 1: total loss 228.1, 34s
  epoch 2: total loss 143.5, 35s
  epoch 3: total loss 90.9, 34s
  fold 1 macro F1: 0.5588
cardiffnlp/twitter-roberta-base fold 2
  epoch 1: total loss 229.4, 34s
  epoch 2: total loss 143.5, 33s
  epoch 3: total loss 86.5, 34s
  fold 2 macro F1: 0.5661
cardiffnlp/twitter-roberta-base fold 3
  epoch 1: total loss 230.5, 34s
  epoch 2: total loss 145.5, 34s
  epoch 3: total loss 94.2, 34s
  fold 3 macro F1: 0.6040
cardiffnlp/twitter-roberta-base fold 4
  epoch 1: total loss 232.4, 35s
  epoch 2: total loss 148.5, 34s
  epoch 3: total loss 89.8, 34s
  fold 4 macro F1: 0.6142
cardiffnlp/twitter-roberta-base fold 5
  epoch 1: total loss 233.3, 33s
  epoch 2: total loss 148.7, 33s
  epoch 3: total loss 94.2, 34s
  fold 5 macro F1: 0.6056
cardiffnlp/twitter-roberta-base 5-fold mean: 0.5897  std: 0.0227
twitter-roberta             OOF macro F1: 0.5907
distil + roberta (sub6)     OOF macro F1: 0.5983
distil + roberta + twi

In [21]:
# ===== one change only: twitter-roberta with 4 epochs instead of 3 =====
oof_t4, P_t4 = run_5fold("cardiffnlp/twitter-roberta-base", epochs=4)
np.save("/kaggle/working/oof_twitter_roberta_4ep.npy", oof_t4)
np.save("/kaggle/working/test_twitter_roberta_4ep.npy", P_t4)

folds = list(skf.split(texts, y_all))
def fold_scores(P):
    return [f1_score(y_all[va], P[va].argmax(axis=1), average="macro") for _, va in folds]

# 1) single model: 3 epochs vs 4 epochs, fold by fold
s3, s4 = fold_scores(oof_t), fold_scores(oof_t4)
print("single twitter-roberta:")
for k in range(5):
    print(f"  fold {k + 1}: 3 epochs {s3[k]:.4f}   4 epochs {s4[k]:.4f}")
print(f"  4 epochs better in {sum(b > a for a, b in zip(s3, s4))} of 5 folds\n")

# 2) and 3) ensembles compared with sub7, fold by fold
base = fold_scores(oof_d + oof_r + oof_t)                  # sub7
options = {"replace (d + r + t4)":   oof_d + oof_r + oof_t4,
           "add (d + r + t3 + t4)":  oof_d + oof_r + oof_t + oof_t4}
print(f"sub7 (d + r + t3)        OOF {f1_score(y_all, (oof_d + oof_r + oof_t).argmax(axis=1), average='macro'):.4f}")
for name, o in options.items():
    s = fold_scores(o)
    wins = sum(b > a for a, b in zip(base, s))
    print(f"{name:24s} OOF {f1_score(y_all, o.argmax(axis=1), average='macro'):.4f}"
          f"   beats sub7 in {wins} of 5 folds   {np.round(s, 4)}")

# save both, I only upload the one that passes my rule
save_sub(P_d + P_r + P_t4,        "/kaggle/working/sub8_replace_t4.csv")
save_sub(P_d + P_r + P_t + P_t4,  "/kaggle/working/sub8_add_t4.csv")

cardiffnlp/twitter-roberta-base fold 1
  epoch 1: total loss 228.1, 34s
  epoch 2: total loss 143.5, 35s
  epoch 3: total loss 90.9, 34s
  epoch 4: total loss 47.2, 34s
  fold 1 macro F1: 0.5803
cardiffnlp/twitter-roberta-base fold 2
  epoch 1: total loss 229.4, 34s
  epoch 2: total loss 143.5, 33s
  epoch 3: total loss 86.5, 34s
  epoch 4: total loss 44.3, 33s
  fold 2 macro F1: 0.5399
cardiffnlp/twitter-roberta-base fold 3
  epoch 1: total loss 230.5, 34s
  epoch 2: total loss 145.5, 34s
  epoch 3: total loss 94.2, 34s
  epoch 4: total loss 52.8, 34s
  fold 3 macro F1: 0.5759
cardiffnlp/twitter-roberta-base fold 4
  epoch 1: total loss 232.4, 34s
  epoch 2: total loss 148.5, 34s
  epoch 3: total loss 89.8, 34s
  epoch 4: total loss 44.1, 35s
  fold 4 macro F1: 0.5941
cardiffnlp/twitter-roberta-base fold 5
  epoch 1: total loss 233.3, 33s
  epoch 2: total loss 148.7, 33s
  epoch 3: total loss 94.2, 34s
  epoch 4: total loss 44.9, 32s
  fold 5 macro F1: 0.5866
cardiffnlp/twitter-robert

In [22]:
# ===== test roberta-large on fold 1 only (big model: smaller lr + warmup + clipping) =====
from transformers import get_linear_schedule_with_warmup

def train_model_safe(texts, labels, epochs=3, lr=1e-5, seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABELS)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = epochs * int(np.ceil(len(texts) / 16))
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)   # warm up, then slow down
    model.train()
    for ep in range(epochs):
        t0, total = time.time(), 0.0
        for enc, y in batches(texts, labels, shuffle=True):
            out = model(**enc, labels=y)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)          # stop huge updates
            opt.step()
            sched.step()
            opt.zero_grad()
            total += out.loss.item()
        print(f"  epoch {ep + 1}: total loss {total:.1f}, {time.time() - t0:.0f}s")
    return model

MODEL_NAME = "roberta-large"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

tr_i, va_i = next(iter(skf.split(texts, y_all)))      # fold 1 only
m = train_model_safe([texts[i] for i in tr_i], y_all[tr_i].tolist(), seed=42)
p = predict_proba(m, [texts[i] for i in va_i])

print("NaN in predictions:", np.isnan(p).any())
print("fold 1 macro F1:", round(f1_score(y_all[va_i], p.argmax(axis=1), average="macro"), 4),
      " (distil 0.5672, roberta 0.5695, twitter 0.5588)")
print(pd.Series([LABELS[i] for i in p.argmax(axis=1)]).value_counts())
del m
torch.cuda.empty_cache()

  epoch 1: total loss 291.9, 113s
  epoch 2: total loss 201.9, 115s
  epoch 3: total loss 151.7, 114s
NaN in predictions: False
fold 1 macro F1: 0.5692  (distil 0.5672, roberta 0.5695, twitter 0.5588)
joy         186
anger       156
sadness     135
surprise     80
disgust      71
fear         44
Name: count, dtype: int64


In [23]:
# ===== Model D: roberta-large, 5 folds, with the safe recipe (lr 1e-5 + warmup + clipping) =====
def run_5fold_safe(model_name, epochs=3, lr=1e-5):
    global tok, MODEL_NAME
    MODEL_NAME = model_name
    tok = AutoTokenizer.from_pretrained(model_name)
    oof_m = np.zeros((len(texts), len(LABELS)))
    P_m = np.zeros((len(test_texts), len(LABELS)))
    f1s = []
    for k, (tr_i, va_i) in enumerate(skf.split(texts, y_all)):
        print(f"{model_name} fold {k + 1}")
        m = train_model_safe([texts[i] for i in tr_i], y_all[tr_i].tolist(),
                             epochs=epochs, lr=lr, seed=42 + k)
        oof_m[va_i] = predict_proba(m, [texts[i] for i in va_i])
        P_m += predict_proba(m, test_texts) / 5
        f1s.append(f1_score(y_all[va_i], oof_m[va_i].argmax(axis=1), average="macro"))
        print(f"  fold {k + 1} macro F1: {f1s[-1]:.4f}")
        del m
        torch.cuda.empty_cache()
    print(f"{model_name} 5-fold mean: {np.mean(f1s):.4f}  std: {np.std(f1s):.4f}\n")
    return oof_m, P_m

oof_L, P_L = run_5fold_safe("roberta-large")
np.save("/kaggle/working/oof_roberta_large.npy", oof_L)
np.save("/kaggle/working/test_roberta_large.npy", P_L)

folds = list(skf.split(texts, y_all))
def fold_scores(P):
    return [f1_score(y_all[va], P[va].argmax(axis=1), average="macro") for _, va in folds]

# single model compared with the other three
for name, o in [("distilroberta", oof_d), ("roberta-base", oof_r),
                ("twitter-roberta", oof_t), ("roberta-large", oof_L)]:
    print(f"{name:16s} OOF macro F1: {f1_score(y_all, o.argmax(axis=1), average='macro'):.4f}")

# does adding roberta-large to sub7's 3 models help, fold by fold?
base = fold_scores(oof_d + oof_r + oof_t)                     # sub7
new = fold_scores(oof_d + oof_r + oof_t + oof_L)
print(f"\nsub7 (d + r + t)       OOF {f1_score(y_all, (oof_d + oof_r + oof_t).argmax(axis=1), average='macro'):.4f}")
print(f"add large (d + r + t + L) OOF {f1_score(y_all, (oof_d + oof_r + oof_t + oof_L).argmax(axis=1), average='macro'):.4f}")
for k in range(5):
    print(f"  fold {k + 1}: sub7 {base[k]:.4f}   + large {new[k]:.4f}")
print(f"adding roberta-large beats sub7 in {sum(b > a for a, b in zip(base, new))} of 5 folds\n")

save_sub(P_d + P_r + P_t + P_L, "/kaggle/working/sub8_add_roberta_large.csv")

roberta-large fold 1
  epoch 1: total loss 291.9, 113s
  epoch 2: total loss 201.9, 116s
  epoch 3: total loss 151.7, 114s
  fold 1 macro F1: 0.5692
roberta-large fold 2
  epoch 1: total loss 290.3, 114s
  epoch 2: total loss 220.4, 112s
  epoch 3: total loss 169.8, 112s
  fold 2 macro F1: 0.5626
roberta-large fold 3
  epoch 1: total loss 281.9, 114s
  epoch 2: total loss 193.4, 114s
  epoch 3: total loss 150.4, 114s
  fold 3 macro F1: 0.6211
roberta-large fold 4
  epoch 1: total loss 285.3, 115s
  epoch 2: total loss 200.0, 113s
  epoch 3: total loss 152.5, 114s
  fold 4 macro F1: 0.6157
roberta-large fold 5
  epoch 1: total loss 293.6, 110s
  epoch 2: total loss 212.6, 111s
  epoch 3: total loss 155.7, 112s
  fold 5 macro F1: 0.5931
roberta-large 5-fold mean: 0.5923  std: 0.0236
distilroberta    OOF macro F1: 0.5784
roberta-base     OOF macro F1: 0.5879
twitter-roberta  OOF macro F1: 0.5907
roberta-large    OOF macro F1: 0.5927
sub7 (d + r + t)       OOF 0.6059
add large (d + r + t +

#### Last idea in 3.2: remove the weakest model (sub9)
distilroberta had the lowest OOF score (0.5784). So I tried using only roberta-base + twitter-roberta + roberta-large.
I ran this cell in a later Kaggle session, and its output was not saved.
The result is computed again in 3.4 (part 2): sub9 OOF 0.6135, and it beats sub8 in 4 of 5 folds.

In [ ]:
# ===== one idea: drop the weakest model (distilroberta) -> sub9 =====
cands = {
    "sub7 (d + r + t)":       (oof_d + oof_r + oof_t,         P_d + P_r + P_t),
    "sub8 (d + r + t + L)":   (oof_d + oof_r + oof_t + oof_L, P_d + P_r + P_t + P_L),
    "no distil (r + t + L)":  (oof_r + oof_t + oof_L,         P_r + P_t + P_L),
}
base = fold_scores(cands["sub8 (d + r + t + L)"][0])
for name, (o, _) in cands.items():
    s = fold_scores(o)
    wins = sum(b > a for a, b in zip(base, s))
    print(f"{name:24s} OOF {f1_score(y_all, o.argmax(axis=1), average='macro'):.4f}"
          f"   beats sub8 in {wins} of 5 folds   {np.round(s, 4)}")

# how many test posts change compared with sub7 and sub8?
p9 = (P_r + P_t + P_L).argmax(axis=1)
print("\ntest posts different from sub7:", (p9 != (P_d + P_r + P_t).argmax(axis=1)).sum())
print("test posts different from sub8:", (p9 != (P_d + P_r + P_t + P_L).argmax(axis=1)).sum(), "\n")

save_sub(P_r + P_t + P_L, "/kaggle/working/sub9_no_distil.csv")


### 3.3 Final models, part 2: new seeds and a bigger tweet model (ran on Kaggle GPU)

I ran these cells on **Kaggle**, in my Kaggle notebook `dmLab2_try02`. **Please do not run them here** (they need a GPU).

- I did **not** train the 4 models from 3.2 again. Cell 3 loads their saved probabilities from my private Kaggle dataset.
- Each experiment changes **only one thing** and uses the **same 5 folds**.
- **My rule (decided before each run):** I upload a new idea only if it beats my current best in **4 or 5 of the 5 folds**.
  Why: a small rise in the OOF score is not enough. For example, sub8 had a higher OOF than sub7, but a lower Public score.

**About the outputs:**
- The [ ] next to these cells is empty because they did not run in this notebook. The outputs are from Kaggle.
- Cells 1-3 and experiments 4-5 show the real Kaggle output (I only removed loading messages, warnings and progress bars).
- Experiments 1-3 ran in earlier Kaggle sessions, and their output was not saved.
  But their scores are computed only from the saved probabilities, so **3.4 computes all of them again here**, and they are the same as on Kaggle.
- 3.4 cannot compute the training loss again, so I do not give any loss numbers for experiments 1-3.

**More short names:**
- **seed**: a starting number for the random parts of training. The same model with a different seed gives slightly different answers.
- **t2, L2, TL2**: the same model trained again with seeds 142-146 (the first time I used 42-46).
- **TL**: twitter-roberta-large.

#### Setup: load the data, the helper functions, and the saved probabilities

In [ ]:
# ===== try02 / 1. Load data (same as try01) =====
import os, json, time
import numpy as np
import pandas as pd
import torch

# find the competition folder automatically
DATA_DIR = [r for r, d, f in os.walk("/kaggle/input") if "final_posts.json" in f][0] + "/"
with open(DATA_DIR + "final_posts.json", encoding="utf-8") as f:
    raw = json.load(f)
emo = pd.read_csv(DATA_DIR + "emotion.csv")
ident = pd.read_csv(DATA_DIR + "data_identification.csv")
sample = pd.read_csv(DATA_DIR + "sampleSubmission.csv")

posts = pd.DataFrame(raw)
df = posts.merge(ident, on="post_id")
train_df = df[df["set"] == "train"].merge(emo, on="post_id")
test_df = df[df["set"] == "test"].reset_index(drop=True)
print("train:", train_df.shape, " test:", test_df.shape)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")
if device == "cpu":
    print("WARNING: no GPU. Turn on GPU T4 in Session options, then restart and run again.")

train: (3360, 4)  test: (1440, 3)
device: cuda Tesla T4


In [ ]:
# ===== try02 / 2. Helpers: everything from try01 in one place =====
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

LABELS = sorted(train_df["label"].unique())                         # the 6 emotions
lab2id = {l: i for i, l in enumerate(LABELS)}
texts = train_df["text"].tolist()
y_all = np.array([lab2id[l] for l in train_df["label"]])
test_texts = test_df["text"].tolist()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)    # same folds as always
folds = list(skf.split(texts, y_all))
tok, MODEL_NAME = None, None                                        # set inside run_5fold()

def batches(texts, labels=None, bs=16, shuffle=False):
    # small batches, each padded only to its own longest post, then moved to the GPU
    idx = np.random.permutation(len(texts)) if shuffle else np.arange(len(texts))
    for s in range(0, len(idx), bs):
        b = idx[s:s + bs]
        enc = tok([texts[i] for i in b], truncation=True, max_length=128,
                  padding=True, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        y = None if labels is None else torch.tensor([labels[i] for i in b]).to(device)
        yield enc, y

def train_model(texts, labels, epochs=3, lr=3e-5, seed=42, safe=False):
    # safe=True: warmup + gradient clipping (the recipe I used for roberta-large)
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABELS)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = epochs * int(np.ceil(len(texts) / 16))
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps) if safe else None
    model.train()
    for ep in range(epochs):
        t0, total = time.time(), 0.0
        for enc, y in batches(texts, labels, shuffle=True):
            out = model(**enc, labels=y)
            out.loss.backward()
            if safe:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            if safe:
                sched.step()
            opt.zero_grad()
            total += out.loss.item()
        print(f"  epoch {ep + 1}: total loss {total:.1f}, {time.time() - t0:.0f}s")
    return model

def predict_proba(model, texts):
    # the model's confidence for each of the 6 emotions (each row adds up to 1)
    model.eval()
    out = []
    with torch.no_grad():
        for enc, _ in batches(texts, bs=64):
            out.append(torch.softmax(model(**enc).logits, dim=-1).cpu().numpy())
    return np.vstack(out)

def oof_f1(P):
    # macro F1 on all train posts, each predicted by a model that never saw it
    return f1_score(y_all, P.argmax(axis=1), average="macro")

def fold_scores(P):
    # macro F1 of each of the 5 folds
    return [f1_score(y_all[va], P[va].argmax(axis=1), average="macro") for _, va in folds]

def run_5fold(model_name, epochs=3, lr=3e-5, seed_base=42, safe=False):
    # 5 models on the same 5 folds -> out-of-fold probs + averaged test probs
    global tok, MODEL_NAME
    MODEL_NAME = model_name
    tok = AutoTokenizer.from_pretrained(model_name)
    oof_m = np.zeros((len(texts), len(LABELS)))
    P_m = np.zeros((len(test_texts), len(LABELS)))
    for k, (tr_i, va_i) in enumerate(folds):
        print(f"{model_name} | seed {seed_base + k} | fold {k + 1}")
        m = train_model([texts[i] for i in tr_i], y_all[tr_i].tolist(),
                        epochs=epochs, lr=lr, seed=seed_base + k, safe=safe)
        oof_m[va_i] = predict_proba(m, [texts[i] for i in va_i])
        P_m += predict_proba(m, test_texts) / 5
        print(f"  fold {k + 1} macro F1: "
              f"{f1_score(y_all[va_i], oof_m[va_i].argmax(axis=1), average='macro'):.4f}")
        del m
        torch.cuda.empty_cache()
    print(f"{model_name} OOF macro F1: {oof_f1(oof_m):.4f}  folds {np.round(fold_scores(oof_m), 4)}\n")
    return oof_m, P_m

def compare(base_name, base_oof, new_name, new_oof):
    # paired check on the same 5 folds; returns how many folds the new one wins
    b, n = fold_scores(base_oof), fold_scores(new_oof)
    wins = sum(y > x for x, y in zip(b, n))
    print(f"{base_name:38s} OOF {oof_f1(base_oof):.4f}  {np.round(b, 4)}")
    print(f"{new_name:38s} OOF {oof_f1(new_oof):.4f}  {np.round(n, 4)}")
    print(f"-> new beats base in {wins} of 5 folds\n")
    return wins

def save_sub(P, path):
    # follow sampleSubmission: same ids and order, the column must be called "id"
    pred = [LABELS[i] for i in P.argmax(axis=1)]
    tmp = pd.DataFrame({"post_id": test_df["post_id"], "label": pred})
    s = sample[["post_id"]].merge(tmp, on="post_id", how="left").rename(columns={"post_id": "id"})
    print(path, s.shape, " missing:", s["label"].isna().sum())
    print(s["label"].value_counts(), "\n")
    s.to_csv(path, index=False)

print("ready, labels:", LABELS)

ready, labels: ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']


In [ ]:
# ===== try02 / 3. Load the 4 models from try01 (saved in my dataset dm-lab2-probs-all) =====
found = [r for r, d, f in os.walk("/kaggle/input") if "oof_roberta_large.npy" in f]
if not found:
    raise FileNotFoundError("Cannot find my .npy files. Add the dataset dm-lab2-probs-all in Input (+ Add Input), then run this cell again.")
SAVED = found[0] + "/"
def load(name):
    return np.load(SAVED + name)

oof_d, P_d = load("oof_distilroberta.npy"),   load("test_distilroberta.npy")
oof_r, P_r = load("oof_roberta_base.npy"),    load("test_roberta_base.npy")
oof_t, P_t = load("oof_twitter_roberta.npy"), load("test_twitter_roberta.npy")
oof_L, P_L = load("oof_roberta_large.npy"),   load("test_roberta_large.npy")

print("saved folder:", SAVED)
print("sub7 check, should be 0.6059:", round(oof_f1(oof_d + oof_r + oof_t), 4))
print("sub9 check, should be 0.6135:", round(oof_f1(oof_r + oof_t + oof_L), 4))

# experiment results saved later (only loaded if I added them to the dataset)
def load_if_there(name):
    hit = [r for r, d, f in os.walk("/kaggle/input") if name in f]
    return np.load(hit[0] + "/" + name) if hit else None

oof_L2, P_L2 = load_if_there("oof_roberta_large_seed2.npy"), load_if_there("test_roberta_large_seed2.npy")
oof_t2, P_t2 = load_if_there("oof_twitter_roberta_seed2.npy"), load_if_there("test_twitter_roberta_seed2.npy")
if oof_L2 is not None:
    print("sub11 check, should be 0.6157:", round(oof_f1(oof_r + oof_t + (oof_L + oof_L2) / 2), 4))
else:
    print("roberta-large seed 2 files not found: add them to dm-lab2-probs-all before experiment 3")

oof_TL, P_TL = load_if_there("oof_twitter_roberta_large.npy"), load_if_there("test_twitter_roberta_large.npy")
if oof_L2 is not None and oof_TL is not None:
    print("sub12 check, should be 0.6220:", round(oof_f1(oof_r + oof_t + (oof_L + oof_L2) / 2 + oof_TL), 4))
else:
    print("twitter-roberta-large files not found: add them to dm-lab2-probs-all before experiments 4 and 5")

saved folder: /kaggle/input/datasets/olpolpolp/dm-lab2-probs-all/
sub7 check, should be 0.6059: 0.6059
sub9 check, should be 0.6135: 0.6135
sub11 check, should be 0.6157: 0.6157
sub12 check, should be 0.6220: 0.622


#### Experiment 1: twitter-roberta with new seeds (11/16)

**Why:** in 3.1, two models with the same settings gave the same answer on only 74.9% of the test posts.
Also, when I averaged more distilroberta models, my Public score kept going up.
So maybe training the same model again with other seeds, and averaging, makes it more stable.

**One change:** train twitter-roberta again with seeds 142-146.
- sub9 = r + t + L
- sub10 = r + avg(t, t2) + L (I divide by 2, so twitter still counts as one model)

**Result (computed again in 3.4):** sub10 OOF 0.6165 vs sub9 0.6135, but it wins only **3 of 5 folds**, so I did **not upload** it.

In [ ]:
# ===== try02 / experiment 1: twitter-roberta with new seeds (142-146), one change only =====
# (output not kept on Kaggle; the scores are recomputed in 3.4)
oof_t2, P_t2 = run_5fold("cardiffnlp/twitter-roberta-base", seed_base=142)
np.save("/kaggle/working/oof_twitter_roberta_seed2.npy", oof_t2)
np.save("/kaggle/working/test_twitter_roberta_seed2.npy", P_t2)

print("old seeds vs new seeds agree on",
      round((oof_t.argmax(axis=1) == oof_t2.argmax(axis=1)).mean() * 100, 1), "% of train posts\n")

compare("twitter seeds 42-46 (t)",    oof_t,
        "twitter seeds 142-146 (t2)", oof_t2)
compare("sub9  (r + t + L)",          oof_r + oof_t + oof_L,
        "sub10 (r + avg(t, t2) + L)", oof_r + (oof_t + oof_t2) / 2 + oof_L)

save_sub(P_r + (P_t + P_t2) / 2 + P_L, "/kaggle/working/sub10_twitter_2seeds.csv")

#### Experiment 2: roberta-large with new seeds (11/17)

**Why:** roberta-large was my best single model at that time (OOF 0.5927), so I tried the same seed idea on it.

**One change:** train roberta-large again with seeds 142-146, with the same safe setting as in 3.2 (learning rate 1e-5, warmup, gradient clipping).
- sub11 = r + t + avg(L, L2), compared with sub9

**Result (computed again in 3.4):** sub11 OOF 0.6157 vs sub9 0.6135, wins **4 of 5 folds**, so I uploaded it as **sub11**. Public 0.6088.

In [ ]:
# ===== try02 / experiment 2: roberta-large with new seeds (142-146), same safe recipe =====
# (output not kept on Kaggle; the scores are recomputed in 3.4)
oof_L2, P_L2 = run_5fold("roberta-large", lr=1e-5, seed_base=142, safe=True)
np.save("/kaggle/working/oof_roberta_large_seed2.npy", oof_L2)
np.save("/kaggle/working/test_roberta_large_seed2.npy", P_L2)

print("old seeds vs new seeds agree on",
      round((oof_L.argmax(axis=1) == oof_L2.argmax(axis=1)).mean() * 100, 1), "% of train posts\n")

compare("roberta-large seeds 42-46 (L)",    oof_L,
        "roberta-large seeds 142-146 (L2)", oof_L2)
compare("sub9  (r + t + L)",                oof_r + oof_t + oof_L,
        "sub11 (r + t + avg(L, L2))",       oof_r + oof_t + (oof_L + oof_L2) / 2)

save_sub(P_r + P_t + (P_L + P_L2) / 2, "/kaggle/working/sub11_large_2seeds.csv")

#### Experiment 3: twitter-roberta-large (11/17)

**Model:** `cardiffnlp/twitter-roberta-large-2022-154m`. It is a RoBERTa-large model that kept learning from 154 million tweets (2018-2022).
It only learned the language (by filling in missing words). It did **not** learn emotion labels.

**Why:** I saw two things:
- Among the smaller (base-size) models, the one that learned from tweets (twitter-roberta) was the best.
- Among all single models, the big one (roberta-large) was the best.

So a big model that learned from tweets might be even better. I used the same safe setting as roberta-large.

**3-1, fold 1 only:** a quick test that training works (no `nan`, not the same label for every post), before spending about 35 minutes on 5 folds.
Its output was not saved. Training worked, and the 5-fold score in 3.4 shows it (TL OOF 0.6114).

**3-2, all 5 folds (computed again in 3.4):** TL alone has OOF **0.6114**, my best single model.
- sub12 = sub11 + TL: OOF 0.6220 vs sub11 0.6157, wins **4 of 5 folds**, so I uploaded it as **sub12**. Public **0.6170** (my best Public score).

In [ ]:
# ===== try02 / experiment 3-1: twitter-roberta-large, fold 1 only (is it broken?) =====
# (output not kept on Kaggle)
MODEL_NAME = "cardiffnlp/twitter-roberta-large-2022-154m"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tr_i, va_i = folds[0]
m = train_model([texts[i] for i in tr_i], y_all[tr_i].tolist(), lr=1e-5, seed=42, safe=True)
p = predict_proba(m, [texts[i] for i in va_i])
print("NaN in predictions:", np.isnan(p).any())
print("fold 1 macro F1:", round(f1_score(y_all[va_i], p.argmax(axis=1), average="macro"), 4),
      " (distil 0.5672, roberta 0.5695, twitter 0.5588, large 0.5692)")
print(pd.Series([LABELS[i] for i in p.argmax(axis=1)]).value_counts())
del m
torch.cuda.empty_cache()

In [ ]:
# ===== try02 / experiment 3-2: twitter-roberta-large, 5 folds, then add it to sub11 =====
# (output not kept on Kaggle; the scores are recomputed in 3.4)
oof_TL, P_TL = run_5fold("cardiffnlp/twitter-roberta-large-2022-154m", lr=1e-5, seed_base=42, safe=True)
np.save("/kaggle/working/oof_twitter_roberta_large.npy", oof_TL)
np.save("/kaggle/working/test_twitter_roberta_large.npy", P_TL)

# single models
for name, o in [("twitter-roberta (t)", oof_t), ("roberta-large (L)", oof_L),
                ("twitter-roberta-large (TL)", oof_TL)]:
    print(f"{name:28s} OOF macro F1: {oof_f1(o):.4f}")
print()

compare("sub11 (r + t + avg(L, L2))",      oof_r + oof_t + (oof_L + oof_L2) / 2,
        "sub12 (r + t + avg(L, L2) + TL)", oof_r + oof_t + (oof_L + oof_L2) / 2 + oof_TL)

save_sub(P_r + P_t + (P_L + P_L2) / 2 + P_TL, "/kaggle/working/sub12_add_twitter_large.csv")

#### Experiment 4: count twitter-roberta-large two times (no training)

**Why:** TL alone (0.6114) was clearly better than the other models (about 0.59). So maybe TL should count more.

To be honest: I only know TL is the best because I looked at the OOF scores, so this idea already uses those scores a little.
So I tried **only one fixed number (x2)**. I did not search for the best number, because in 3.1, numbers I tuned this way did worse on new data (0 of 5 folds).

**Result (real output below):** sub13 OOF 0.6189 vs sub12 0.6220, wins only **2 of 5 folds**, so I did **not upload** it.

In [ ]:
# ===== try02 / experiment 4: count twitter-roberta-large twice (no training) =====
RUN_THIS = True
if not RUN_THIS:
    print("experiment 4 skipped")
elif oof_TL is None or oof_L2 is None:
    print("stop: twitter-roberta-large or roberta-large seed 2 files are not loaded (see cell 3).")
else:
    compare("sub12 (r + t + avg(L, L2) + TL)",     oof_r + oof_t + (oof_L + oof_L2) / 2 + oof_TL,
            "sub13 (r + t + avg(L, L2) + 2 x TL)", oof_r + oof_t + (oof_L + oof_L2) / 2 + 2 * oof_TL)
    save_sub(P_r + P_t + (P_L + P_L2) / 2 + 2 * P_TL, "/kaggle/working/sub13_tl_double.csv")

sub12 (r + t + avg(L, L2) + TL)        OOF 0.6220  [0.6041 0.5975 0.6391 0.6576 0.6089]
sub13 (r + t + avg(L, L2) + 2 x TL)    OOF 0.6189  [0.5981 0.6003 0.642  0.6443 0.6073]
-> new beats base in 2 of 5 folds

/kaggle/working/sub13_tl_double.csv (1440, 2)  missing: 0
label
joy         385
anger       334
sadness     284
surprise    174
disgust     157
fear        106
Name: count, dtype: int64 



#### Experiment 5: twitter-roberta-large with new seeds

**Why:** for roberta-large, averaging two seed sets passed my rule (experiment 2). So I tried the same thing on TL, my best model.

**One change:** train TL again with seeds 142-146.
- sub14 = r + t + avg(L, L2) + avg(TL, TL2), compared with sub12 (not with experiment 4)

**Result (real output below):** TL2 alone OOF 0.6139. sub14 OOF **0.6268** vs sub12 0.6220, wins **4 of 5 folds**, so I uploaded it as **sub14**. Public 0.6111.

In [ ]:
# ===== try02 / experiment 5: twitter-roberta-large with new seeds (142-146) =====
RUN_THIS = True
if not RUN_THIS:
    print("experiment 5 skipped")
elif oof_TL is None or oof_L2 is None:
    print("stop: twitter-roberta-large or roberta-large seed 2 files are not loaded (see cell 3).")
else:
    oof_TL2, P_TL2 = run_5fold("cardiffnlp/twitter-roberta-large-2022-154m", lr=1e-5, seed_base=142, safe=True)
    np.save("/kaggle/working/oof_twitter_roberta_large_seed2.npy", oof_TL2)
    np.save("/kaggle/working/test_twitter_roberta_large_seed2.npy", P_TL2)

    print("old seeds vs new seeds agree on",
          round((oof_TL.argmax(axis=1) == oof_TL2.argmax(axis=1)).mean() * 100, 1), "% of train posts\n")

    compare("TL seeds 42-46 (TL)",                    oof_TL,
            "TL seeds 142-146 (TL2)",                 oof_TL2)
    compare("sub12 (r + t + avg(L, L2) + TL)",        oof_r + oof_t + (oof_L + oof_L2) / 2 + oof_TL,
            "sub14 (r + t + avg(L, L2) + avg(TL, TL2))", oof_r + oof_t + (oof_L + oof_L2) / 2 + (oof_TL + oof_TL2) / 2)

    save_sub(P_r + P_t + (P_L + P_L2) / 2 + (P_TL + P_TL2) / 2, "/kaggle/working/sub14_tl_2seeds.csv")

cardiffnlp/twitter-roberta-large-2022-154m | seed 142 | fold 1
  epoch 1: total loss 269.8, 122s
  epoch 2: total loss 163.8, 117s
  epoch 3: total loss 119.7, 115s
  fold 1 macro F1: 0.6048
cardiffnlp/twitter-roberta-large-2022-154m | seed 143 | fold 2
  epoch 1: total loss 264.9, 116s
  epoch 2: total loss 168.0, 118s
  epoch 3: total loss 123.8, 115s
  fold 2 macro F1: 0.5945
cardiffnlp/twitter-roberta-large-2022-154m | seed 144 | fold 3
  epoch 1: total loss 272.1, 115s
  epoch 2: total loss 180.9, 113s
  epoch 3: total loss 130.3, 116s
  fold 3 macro F1: 0.6446
cardiffnlp/twitter-roberta-large-2022-154m | seed 145 | fold 4
  epoch 1: total loss 277.7, 117s
  epoch 2: total loss 184.2, 117s
  epoch 3: total loss 138.9, 118s
  fold 4 macro F1: 0.6326
cardiffnlp/twitter-roberta-large-2022-154m | seed 146 | fold 5
  epoch 1: total loss 267.2, 114s
  epoch 2: total loss 170.4, 116s
  epoch 3: total loss 123.5, 114s
  fold 5 macro F1: 0.5906
cardiffnlp/twitter-roberta-large-2022-154m OO

### 3.4 Check all Kaggle scores again, here (laptop CPU, a few seconds, real output)

These two cells ran in this notebook. They load the 16 probability files I saved on Kaggle (`submissions/*.npy`),
make the same 5 folds again, and compute the scores again **without any training**.

- Part 1 checks 3.3: every OOF score says `same`, and every "wins x of 5 folds" is the same number Kaggle printed.
- Part 2 checks 3.2 (sub5 to sub9) in the same way.

In [1]:
# ===== 3.4 Re-check all Kaggle results from the saved probabilities (CPU, a few seconds) =====
# On Kaggle I did not keep the printed output of some runs (sub9, experiments 1-3).
# But every score in my log comes only from the saved .npy probabilities, so I can
# compute all of them again here, with no training.
# Same steps as on Kaggle: same merge (same post order), same 5 folds.
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

KG_DATA = "data/kaggle/"      # competition files (same as the top of this notebook)
KG_PROBS = "submissions/"     # the 16 .npy files I downloaded from Kaggle

# rebuild the train / test posts in the same order as on Kaggle
with open(KG_DATA + "final_posts.json", encoding="utf-8") as f:
    kg_posts = pd.DataFrame(json.load(f))
kg_df = kg_posts.merge(pd.read_csv(KG_DATA + "data_identification.csv"), on="post_id")
kg_train = kg_df[kg_df["set"] == "train"].merge(pd.read_csv(KG_DATA + "emotion.csv"), on="post_id")
kg_labels = sorted(kg_train["label"].unique())
kg_y = np.array([kg_labels.index(l) for l in kg_train["label"]])
kg_folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                .split(kg_train["text"].tolist(), kg_y))
print("train posts:", len(kg_y), " labels:", kg_labels, "\n")

def kg_load(name):
    # OOF probs (3360 x 6) and test probs (1440 x 6) of one model
    return (np.load(KG_PROBS + "oof_" + name + ".npy"),
            np.load(KG_PROBS + "test_" + name + ".npy"))

def kg_f1(P):
    # macro F1 on all 3360 train posts (each predicted by a model that never saw it)
    return f1_score(kg_y, P.argmax(axis=1), average="macro")

def kg_fold_f1(P):
    # macro F1 of each of the 5 folds
    return [f1_score(kg_y[va], P[va].argmax(axis=1), average="macro") for _, va in kg_folds]

def kg_compare(step, base_name, base, new_name, new, kaggle_wins):
    # paired check on the same 5 folds, next to what Kaggle printed
    b, n = kg_fold_f1(base), kg_fold_f1(new)
    wins = sum(y > x for x, y in zip(b, n))
    print(step)
    print(f"  {base_name:40s} OOF {kg_f1(base):.4f}  {np.round(b, 4)}")
    print(f"  {new_name:40s} OOF {kg_f1(new):.4f}  {np.round(n, 4)}")
    print(f"  -> new wins {wins} of 5 folds (Kaggle: {kaggle_wins})\n")

oof_d,   _ = kg_load("distilroberta")
oof_r,   _ = kg_load("roberta_base")
oof_t,   _ = kg_load("twitter_roberta")
oof_t2,  _ = kg_load("twitter_roberta_seed2")
oof_L,   _ = kg_load("roberta_large")
oof_L2,  _ = kg_load("roberta_large_seed2")
oof_TL,  P_TL  = kg_load("twitter_roberta_large")
oof_TL2, P_TL2 = kg_load("twitter_roberta_large_seed2")

# 1) single models (5-fold OOF)
print("single models:")
for name, o in [("distilroberta (d)", oof_d), ("roberta-base (r)", oof_r),
                ("twitter-roberta (t)", oof_t), ("twitter-roberta new seeds (t2)", oof_t2),
                ("roberta-large (L)", oof_L), ("roberta-large new seeds (L2)", oof_L2),
                ("twitter-roberta-large (TL)", oof_TL), ("twitter-roberta-large new seeds (TL2)", oof_TL2)]:
    print(f"  {name:40s} OOF {kg_f1(o):.4f}")
print()

# 2) ensembles: my numbers vs the OOF that Kaggle printed
sub9  = oof_r + oof_t + oof_L
sub10 = oof_r + (oof_t + oof_t2) / 2 + oof_L
sub11 = oof_r + oof_t + (oof_L + oof_L2) / 2
sub12 = sub11 + oof_TL
sub13 = sub11 + 2 * oof_TL
sub14 = sub11 + (oof_TL + oof_TL2) / 2
print("ensembles (OOF here vs OOF on Kaggle):")
for name, P, kaggle in [("sub7  d + r + t", oof_d + oof_r + oof_t, 0.6059),
                        ("sub9  r + t + L", sub9, 0.6135),
                        ("sub10 r + avg(t, t2) + L", sub10, 0.6165),
                        ("sub11 r + t + avg(L, L2)", sub11, 0.6157),
                        ("sub12 sub11 + TL", sub12, 0.6220),
                        ("sub13 sub11 + 2 x TL", sub13, 0.6189),
                        ("sub14 sub11 + avg(TL, TL2)", sub14, 0.6268)]:
    here = round(kg_f1(P), 4)
    print(f"  {name:28s} {here:.4f}  Kaggle {kaggle:.4f}  {'same' if here == kaggle else 'DIFFERENT'}")
print()

# 3) the paired fold checks I used to decide what to upload (rule: win 4-5 of 5 folds)
kg_compare("experiment 1: twitter-roberta new seeds", "sub9  r + t + L", sub9,
           "sub10 r + avg(t, t2) + L", sub10, 3)
kg_compare("experiment 2: roberta-large new seeds", "sub9  r + t + L", sub9,
           "sub11 r + t + avg(L, L2)", sub11, 4)
kg_compare("experiment 3: add twitter-roberta-large", "sub11 r + t + avg(L, L2)", sub11,
           "sub12 sub11 + TL", sub12, 4)
kg_compare("experiment 4: count TL twice", "sub12 sub11 + TL", sub12,
           "sub13 sub11 + 2 x TL", sub13, 2)
kg_compare("experiment 5: twitter-roberta-large new seeds", "sub12 sub11 + TL", sub12,
           "sub14 sub11 + avg(TL, TL2)", sub14, 4)

# 4) how often the old and new seeds pick the same label
for name, a, b, kaggle in [("t vs t2", oof_t, oof_t2, 77.8), ("L vs L2", oof_L, oof_L2, 82.9),
                           ("TL vs TL2", oof_TL, oof_TL2, 86.8)]:
    print(f"{name:10s} agree on {(a.argmax(1) == b.argmax(1)).mean() * 100:.1f} % of train posts (Kaggle {kaggle})")

train posts: 3360  labels: ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise'] 

single models:
  distilroberta (d)                        OOF 0.5784
  roberta-base (r)                         OOF 0.5879
  twitter-roberta (t)                      OOF 0.5907
  twitter-roberta new seeds (t2)           OOF 0.5869
  roberta-large (L)                        OOF 0.5927
  roberta-large new seeds (L2)             OOF 0.5962
  twitter-roberta-large (TL)               OOF 0.6114
  twitter-roberta-large new seeds (TL2)    OOF 0.6139

ensembles (OOF here vs OOF on Kaggle):
  sub7  d + r + t              0.6059  Kaggle 0.6059  same
  sub9  r + t + L              0.6135  Kaggle 0.6135  same
  sub10 r + avg(t, t2) + L     0.6165  Kaggle 0.6165  same
  sub11 r + t + avg(L, L2)     0.6157  Kaggle 0.6157  same
  sub12 sub11 + TL             0.6220  Kaggle 0.6220  same
  sub13 sub11 + 2 x TL         0.6189  Kaggle 0.6189  same
  sub14 sub11 + avg(TL, TL2)   0.6268  Kaggle 0.6268  same

experiment 

Part 2: it uses the functions from part 1, and checks the 3.2 decisions (sub5 to sub9) in the same way.

In [2]:
# ===== 3.4 (part 2) Re-check the 3.2 decisions: sub5 -> sub6 -> sub7 -> sub8 -> sub9 =====
# uses kg_f1, kg_compare and the oof_ arrays from the cell above
sub5 = oof_d
sub6 = oof_d + oof_r
sub7 = oof_d + oof_r + oof_t
sub8 = oof_d + oof_r + oof_t + oof_L
for name, P, kaggle in [("sub5  d", sub5, 0.5784), ("sub6  d + r", sub6, 0.5983),
                        ("sub8  d + r + t + L", sub8, 0.6093)]:
    here = round(kg_f1(P), 4)
    print(f"  {name:28s} {here:.4f}  Kaggle {kaggle:.4f}  {'same' if here == kaggle else 'DIFFERENT'}")
print()
kg_compare("3.2: add roberta-base", "sub5  d", sub5, "sub6  d + r", sub6, 5)
kg_compare("3.2: add twitter-roberta", "sub6  d + r", sub6, "sub7  d + r + t", sub7, 4)
kg_compare("3.2: add roberta-large", "sub7  d + r + t", sub7, "sub8  d + r + t + L", sub8, 5)
kg_compare("3.2: drop distilroberta", "sub8  d + r + t + L", sub8, "sub9  r + t + L", oof_r + oof_t + oof_L, 4)

  sub5  d                      0.5784  Kaggle 0.5784  same
  sub6  d + r                  0.5983  Kaggle 0.5983  same
  sub8  d + r + t + L          0.6093  Kaggle 0.6093  same

3.2: add roberta-base
  sub5  d                                  OOF 0.5784  [0.5672 0.5462 0.6042 0.593  0.5853]
  sub6  d + r                              OOF 0.5983  [0.5836 0.5584 0.6334 0.6123 0.6046]
  -> new wins 5 of 5 folds (Kaggle: 5)

3.2: add twitter-roberta
  sub6  d + r                              OOF 0.5983  [0.5836 0.5584 0.6334 0.6123 0.6046]
  sub7  d + r + t                          OOF 0.6059  [0.587  0.5731 0.6287 0.6282 0.6112]
  -> new wins 4 of 5 folds (Kaggle: 4)

3.2: add roberta-large
  sub7  d + r + t                          OOF 0.6059  [0.587  0.5731 0.6287 0.6282 0.6112]
  sub8  d + r + t + L                      OOF 0.6093  [0.5897 0.5779 0.6325 0.6336 0.6115]
  -> new wins 5 of 5 folds (Kaggle: 5)

3.2: drop distilroberta
  sub8  d + r + t + L                      OOF 0.6093  [

### 3.5 My final submissions

On Kaggle I chose 2 final submissions (Kaggle lets me choose up to 2):

| | Models | OOF (5-fold) | Public |
|---|---|---|---|
| **sub14** | r + t + avg(L, L2) + avg(TL, TL2) | **0.6268** (best) | 0.6111 |
| **sub12** | r + t + avg(L, L2) + TL | 0.6220 | **0.6170** (best) |

**Why two different ones:** OOF uses all 3,360 train posts, but Public uses only part of the test posts.
Two times, my OOF went up but my Public went down:
- sub8 vs sub7: OOF +0.0034, Public -0.0146
- sub14 vs sub12: OOF +0.0048, Public -0.0059

So I could not tell which one is really better. I chose the best of each, and the Private leaderboard decides.

### Experiment & submission log

| ID | Date | Where (code) | Method | Validation | Public | Decision |
|---|---|---|---|---|---|---|
| sub1 | 11/11 | Homework (CPU) | TF-IDF (min_df=1, 1-2 gram) + MiniLM, LinearSVC C=1 | 5-fold CV 0.5313 | 0.5321 | passed baseline |
| sub2 | 11/11 | Homework (CPU) | fine-tuned distilroberta, 100% train, 3 epochs | same recipe: holdout 0.6166 (80% model, one split) | 0.5776 | |
| sub3 | 11/11 | Homework (CPU) | same, 80% train (holdout model) | holdout 0.6166 | 0.5637 | |
| sub4 | 11/11 | Homework (CPU) | average of sub2 + sub3 models | - | 0.5808 | |
| - | 11/11 | Homework (CPU) | mpnet sentence embeddings: swap for MiniLM / add to MiniLM | swap CV 0.5278 / add CV 0.5359 (+0.0046 < std) | - | not uploaded |
| sub5 | 11/12 | Kaggle dmLab2_try01 | 5-fold distilroberta, average of 5 | OOF 0.5784 | 0.5978 | |
| sub6 | 11/12 | Kaggle dmLab2_try01 | + roberta-base | OOF 0.5983, 5/5 folds | 0.6057 | |
| - | 11/12 | Homework (CPU) | class weights tuned on OOF | 0/5 folds | - | not uploaded |
| - | 11/12 | Kaggle dmLab2_try01 | deberta-v3-base | loss nan, all anger | - | dropped |
| sub7 | 11/12 | Kaggle dmLab2_try01 | + twitter-roberta | OOF 0.6059, 4/5 folds | 0.6108 | final candidate until sub12 |
| - | 11/12 | Kaggle dmLab2_try01 | twitter-roberta 4 epochs | 2/5 folds | - | not uploaded |
| sub8 | 11/12 | Kaggle dmLab2_try01 | + roberta-large (lr 1e-5, warmup, clip) | OOF 0.6093, 5/5 folds | 0.5962 | |
| sub9 | 11/12 | Kaggle dmLab2_try01 | roberta-base + twitter + large (no distil) | OOF 0.6135 (sub8 0.6093), 4/5 folds | 0.6060 | final candidate until sub12 |
| - | 11/16 | Kaggle dmLab2_try02 | twitter-roberta new seeds (142-146): r + avg(t, t2) + L | OOF 0.6165 (sub9 0.6135), 3/5 folds | - | not uploaded |
| sub11 | 11/17 | Kaggle dmLab2_try02 | r + t + avg(roberta-large 2 seed sets) | OOF 0.6157 (sub9 0.6135), 4/5 folds | 0.6088 | |
| sub12 | 11/17 | Kaggle dmLab2_try02 | sub11 + twitter-roberta-large (lr 1e-5, warmup, clip) | OOF 0.6220 (sub11 0.6157), 4/5 folds; TL alone 0.6114 | 0.6170 | **final selection (best Public)** |
| - | 11/17 | Kaggle dmLab2_try02 | sub12 with TL counted twice | OOF 0.6189 (sub12 0.6220), 2/5 folds | - | not uploaded |
| sub14 | 11/17 | Kaggle dmLab2_try02 | r + t + avg(L, L2) + avg(TL, TL2) | OOF 0.6268 (sub12 0.6220), 4/5 folds; TL2 alone 0.6139 | 0.6111 | **final selection (best OOF)** |

Every validation number from Kaggle in this table can be found in a real Kaggle output (3.2 / 3.3), or is computed again in 3.4 (most are in both). Public scores come from the Kaggle leaderboard.
